# [PyTorch] Implémentation d'un super routeur qui bascule dynamiquement entre plusieurs modèles
## ~ Expérimentation du routage dur parfait via Gumbel-Softmax ~

Dans ce notebook, nous intégrons plusieurs modèles spécialisés indépendants (modèle 1 spécialisé en français et en allemand et modèle 2 spécialisé en espagnol) et construisons un **Super Router** qui distribue les entrées au modèle approprié.

Nous démontrons le problème du « Lazy Routing » qui se pose avec le routage ordinaire basé sur Softmax (Soft Routing) et confirmons la puissance du Hard Routing parfait (une répartition nette de 100 % contre 0 %) obtenu via **Gumbel-Softmax** pour le résoudre.

In [ ]:
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/JunSuzukiJapan/SynapticRouter.git
    %cd SynapticRouter
    !pip install torch

sys.path.append('.')
sys.path.append('./src')
if 'google.colab' not in sys.modules:
    sys.path.append('..')
    sys.path.append('../src')

import torch
import torch.nn as nn
import torch.nn.functional as F
import random
try:
    from sra_language_models import MoESRALanguageModel
except ImportError:
    from src.sra_language_models import MoESRALanguageModel

torch.manual_seed(42)
random.seed(42)

### 1. Préparation de l'ensemble de données et des utilitaires
Nous préparons un ensemble de données très simple au niveau des mots qui se traduit du français, de l'allemand et de l'espagnol vers l'anglais.

In [ ]:
PARALLEL_DATA = [
    {"eng": "hello", "fra": "bonjour", "deu": "hallo", "spa": "hola"},
    {"eng": "apple", "fra": "pomme", "deu": "apfel", "spa": "manzana"},
    {"eng": "cat", "fra": "chat", "deu": "katze", "spa": "gato"},
    {"eng": "dog", "fra": "chien", "deu": "hund", "spa": "perro"},
    {"eng": "water", "fra": "eau", "deu": "wasser", "spa": "agua"}
]

vocab = ["[PAD]", "[ENG]", "[FRA]", "[DEU]", "[SPA]", "[SEP]", "[EOS]"]
for pair in PARALLEL_DATA:
    for word in pair.values():
        if word not in vocab:
            vocab.append(word)

word2idx = {w: i for i, w in enumerate(vocab)}
idx2word = {i: w for i, w in enumerate(vocab)}
VOCAB_SIZE = len(vocab)

def encode(lang_tag, src_word, tgt_word=None):
    prompt = [word2idx[lang_tag], word2idx[src_word], word2idx["[SEP]"], word2idx["[ENG]"]]
    if tgt_word:
        target = [word2idx[tgt_word], word2idx["[EOS]"]]
        return prompt + target
    return prompt

def get_batch(langs, batch_size=16):
    x = torch.zeros((batch_size, 6), dtype=torch.long)
    y = torch.full((batch_size, 6), -100, dtype=torch.long)
    for i in range(batch_size):
        lang = random.choice(langs)
        lang_tag = f"[{lang.upper()}]"
        pair = random.choice(PARALLEL_DATA)
        seq = encode(lang_tag, pair[lang], pair["eng"])
        x[i, :5] = torch.tensor(seq[:5])
        y[i, 3:5] = torch.tensor(seq[4:6])
    return x, y

### 2. Définir les fonctions de formation et de test
Nous préparons une fonction de test qui visualise le comportement du routeur (avec quelle probabilité chaque langage a été routé vers quel modèle).

In [ ]:
def load_balance_loss(router_logits):
    if not router_logits: return torch.tensor(0.0)
    loss = 0.0
    for logits in router_logits:
        probs = F.softmax(logits, dim=-1)
        mean_probs = probs.mean(dim=(0, 1))
        loss += (mean_probs ** 2).sum() * logits.size(-1)
    return loss

def train_model(model, langs, steps=400, lr=2e-3):
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    model.train()
    for step in range(steps):
        x, y = get_batch(langs)
        logits, router_logits = model(x)
        ce_loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1), ignore_index=-100)
        lb_loss = load_balance_loss(router_logits)
        loss = ce_loss + 0.1 * lb_loss
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    print(f"Trained on {langs} - Final Loss: {loss.item():.4f}")

def test_model(model, langs_to_test, is_super=False):
    model.eval()
    with torch.no_grad():
        for lang in langs_to_test:
            lang_tag = f"[{lang.upper()}]"
            pair = PARALLEL_DATA[0] # Test with 'hello'
            src_word = pair[lang]
            expected = pair["eng"]
            x = torch.tensor([encode(lang_tag, src_word)]).long()
            logits, router_logits = model(x)
            pred_idx = logits[0, 3].argmax().item()
            pred_word = idx2word[pred_idx]
            success = pred_word == expected
            status = "OK" if success else "NG"

            if is_super and router_logits:
                # At test time, display the deterministic probabilities
                top_idx = router_logits[0][0].argmax().item()
                top_model = top_idx + 1
                probs = F.softmax(router_logits[0][0], dim=-1)
                print(f"{status} {lang.upper()} -> ENG: '{src_word}' -> '{pred_word}' | [Routed to Model {top_model}] probs: {[round(p, 3) for p in probs[0].tolist()]}")
            else:
                print(f"{status} {lang.upper()} -> ENG: '{src_word}' -> '{pred_word}'")

### 3. Pré-formation des modèles spécialisés indépendants (Modèle 1, Modèle 2)
Nous préparons deux petits modèles de langage basés sur SRA et formons chacun indépendamment.
- **Modèle 1** : spécialisé en traduction française et allemande
- **Modèle 2** : spécialisé en traduction espagnole

In [ ]:
DIM = 64
model1 = MoESRALanguageModel(vocab_size=VOCAB_SIZE, dim=DIM, layers=1, num_synapses=2, k=1, syn_hidden=128)
model2 = MoESRALanguageModel(vocab_size=VOCAB_SIZE, dim=DIM, layers=1, num_synapses=1, k=1, syn_hidden=128)

print("Training Model 1 (fra, deu)...")
train_model(model1, ["fra", "deu"], steps=600)

print("Training Model 2 (spa)...")
train_model(model2, ["spa"], steps=600)

### 4. Conception et mise en œuvre du super routeur
Nous créons une classe Super Router qui encapsule le modèle 1 et le modèle 2 et décide, en fonction du contexte (la phrase d'entrée entière), quel modèle doit gérer le traitement.
À titre de comparaison, nous mettons en œuvre trois stratégies de routage.

1. **`soft`** : routage progressif. Utilise les probabilités directement comme poids.
2. **`ste`** : Estimateur direct. Sélectionne de force Top-1, mais laisse les dégradés s'écouler comme si vous utilisiez Softmax.
3. **`gumbel`** : Gumbel-Softmax. Ajoute du bruit Gumbel pour effectuer un routage dur qui reste différenciable uniquement pendant l'entraînement.

In [ ]:
class SuperRouterModel(nn.Module):
    def __init__(self, model1, model2, vocab_size, dim, routing_type='soft'):
        super().__init__()
        self.model1 = model1
        self.model2 = model2
        self.routing_type = routing_type

        # Freeze the base models' weights (do not train them)
        for p in self.model1.parameters(): p.requires_grad = False
        for p in self.model2.parameters(): p.requires_grad = False

        # Embedding layer and discriminator for the router
        self.super_embed = nn.Embedding(vocab_size, dim)
        self.super_key = nn.Linear(dim, 2, bias=False)

    def forward(self, x):
        h = self.super_embed(x)
        # Extract sentence-level features via mean pooling
        h_pool = h.mean(dim=1, keepdim=True) # (B, 1, dim)
        router_logits = self.super_key(h_pool) # (B, 1, 2)

        if self.routing_type == 'soft':
            routing_weights = F.softmax(router_logits, dim=-1)

        elif self.routing_type == 'gumbel':
            if self.training:
                routing_weights = F.gumbel_softmax(router_logits, tau=1.0, hard=True, dim=-1)
            else:
                top_idx = router_logits.argmax(dim=-1, keepdim=True)
                routing_weights = torch.zeros_like(router_logits).scatter_(-1, top_idx, 1.0)

        elif self.routing_type == 'ste':
            probs = F.softmax(router_logits, dim=-1)
            top_idx = probs.argmax(dim=-1, keepdim=True)
            hard_weights = torch.zeros_like(probs).scatter_(-1, top_idx, 1.0)
            if self.training:
                routing_weights = hard_weights.detach() - probs.detach() + probs
            else:
                routing_weights = hard_weights

        out1, _ = self.model1(x)
        out2, _ = self.model2(x)

        w1 = routing_weights[..., 0:1]
        w2 = routing_weights[..., 1:2]
        final_out = out1 * w1 + out2 * w2

        return final_out, [router_logits]

### 5. Expériences et comparaison des résultats
Nous entraînons le Super Router avec chaque stratégie de routage et observons comment la distribution de probabilité (confiance) change.

In [ ]:
for routing_type in ['soft', 'ste', 'gumbel']:
    print(f"\n{'='*50}\nTesting Routing Type: {routing_type.upper()}\n{'='*50}")
    torch.manual_seed(42)
    super_model = SuperRouterModel(model1, model2, VOCAB_SIZE, DIM, routing_type=routing_type)

    # Train only the Super Router
    train_model(super_model, ["fra", "deu", "spa"], steps=1000, lr=5e-3)

    # Check the test results (probability distribution)
    test_model(super_model, ["fra", "deu", "spa"], is_super=True)

###Conclusion
- **SOFT** est paresseux (laisse une probabilité résiduelle telle que `0.9 : 0.1`, donc le calcul continue vers des modèles inutiles).
- **STE** a tendance à arrêter d'apprendre une fois qu'il a choisi le bon modèle, et la probabilité reste floue comme `0.6 : 0.4`.
- Avec **GUMBEL**, la robustesse induite par le bruit entre en jeu et nous obtenons un **routage clairsemé `1.0 : 0.0` parfait (supprimant 100 % du calcul pour les modèles inutiles)** !